<a href="https://colab.research.google.com/github/AliRaddman/divar-ml-project/blob/main/notebooks/work/prep_ali.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl">

# آماده‌سازی داده
شامل: اصلاح نوع داده‌ها، تبدیل تاریخ شمسی، و مدیریت مقادیر گم‌شده.

</div>

In [13]:
import pandas as pd
import numpy as np
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [34]:
!pip install jdatetime -q
import jdatetime

In [14]:
df = pd.read_csv('/content/drive/MyDrive/Divar Dataset/Divar.csv')
print(df.shape)
df.head()

/tmp/ipykernel_5178/3751022427.py:1: DtypeWarning: Columns (11,27,29,53) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/content/drive/MyDrive/Divar Dataset/Divar.csv')


(1000000, 61)


,Unnamed: 0,cat2_slug,cat3_slug,city_slug,neighborhood_slug,created_at_month,user_type,description,title,rent_mode,...,property_type,regular_person_capacity,extra_person_capacity,cost_per_extra_person,rent_price_on_regular_days,rent_price_on_special_days,rent_price_at_weekends,location_latitude,location_longitude,location_radius
0,0,temporary-rent,villa,karaj,mehrshahr,2024-08-01 00:00:00,مشاور املاک,۵۰۰متر\n۲۰۰متر بنا دوبلکس\n۳خواب\nاستخر آبگرم ...,باغ ویلا اجاره روزانه استخر داخل لشکرآباد سهیلیه,NaN,...,NaN,4.0,6,350000.0,1500000.0,3.500000e+09,3500000.0,35.811684,50.936600,500.0
1,1,residential-sell,apartment-sell,tehran,gholhak,2024-05-01 00:00:00,مشاور املاک,دسترسی عالی به مترو و شریعتی \nمشاعات تمیز \nب...,۶۰ متر قلهک فول امکانات,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,500.0
2,2,residential-rent,apartment-rent,tehran,tohid,2024-10-01 00:00:00,NaN,تخلیه پایان ماه,آپارتمان ۳ خوابه ۱۳۲ متر,مقطوع,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,35.703865,51.373459,NaN
3,3,commercial-rent,office-rent,tehran,elahiyeh,2024-06-01 00:00:00,NaN,فرشته تاپ لوکیشن\n۹۰ متر موقعیت اداری\nیک اتاق...,فرشته ۹۰ متر دفتر کار مدرن موقعیت اداری,مقطوع,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,residential-sell,apartment-sell,mashhad,emamreza,2024-05-01 00:00:00,مشاور املاک,هلدینگ ساختمانی اکبری\n\nهمراه شما هستیم برای ...,۱۱۵ متری/شمالی رو به آفتاب/اکبری,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<div dir="rtl">

## اصلاح نوع داده‌ها

موقع لود کردن دیتا، خیلی از ستون‌ها نوعشون درست نبود. بعضیا که باید عدد می‌بودن به خاطر ارقام فارسی متنی خونده شده بودن، بعضیا هم اسمشون گول‌زننده بود. اینجا همه رو درست کردیم:

**عددی کردیم:**
- `floor`، `total_floors_count`: مستقیم به عدد
- `rooms_count`: کلمه بود («دو»، «سه»، ...)، نگاشتش کردیم به عدد. «بدون اتاق» شد ۰ و «پنج یا بیشتر» شد ۵
- `unit_per_floor`: عددی بود ولی یه مقدار `more_than_8` داشت که گذاشتیمش ۹
- `construction_year`: ارقامش فارسی بود، انگلیسی کردیم. یه دسته‌ی «قبل از ۱۳۷۰» هم داشت (حدود ۲۰ هزار تا) که گذاشتیمش ۱۳۶۵ — چون این خونه‌ها در واقع قبل از ۱۳۷۰ ساخته شدن. برای آزمون فرض ۲ هم که آستانه‌ش ۱۳۹۶ ـه فرقی نمی‌کنه

**یه کشف مهم:** `transformable_price` و `rent_credit_transform` با اینکه اسمشون قیمت/تبدیله، مقدارشون True/False ـه (فلگن، نه عدد). قیمت واقعی فروش توی `price_value` ـه، رهن توی `credit_value`/`transformable_credit` و اجاره توی `rent_value`. این رو حواسمون باشه برای ساخت قیمت واحد.

**بولین کردیم:** همه‌ی `has_*` ها و `is_rebuilt` و اون دو تا فلگ بالا. `has_balcony` قاطی بود (هم true/false متنی داشت هم بولین) که یکدستش کردیم.

**یه کشف دیگه:** `has_warm_water_provider`، `has_heating_system`، `has_cooling_system`، `has_restroom` با اینکه `has_` دارن، بولین نیستن! اینا نوع سیستم رو نشون می‌دن (شوفاژ، اسپلیت، ...) پس دسته‌ای حسابشون کردیم نه True/False.

**دسته‌ای کردیم:** اسم/دسته‌ها مثل شهر، محله، نوع سند و... رو به `category` تبدیل کردیم که هم حافظه کمتر بگیره هم سریع‌تر شه. `unselect` ها رو هم به NaN تبدیل کردیم چون یعنی کاربر چیزی انتخاب نکرده.

**تاریخ:** `created_at_month` میلادی بود، با jdatetime به شمسی تبدیلش کردیم. سه نسخه ساختیم: خود تاریخ میلادی (برای مرتب‌کردن)، فرمت `1403-04` (برای گروه‌بندی) و فرمت خوانا «تیر 1403» (برای نمودار).

ستون اضافی `Unnamed: 0` هم که یه ایندکس بی‌خود بود حذف شد.

</div>

In [15]:
# نوع داده‌ی همه‌ی ستون‌ها
print(df.dtypes)

print('\n===== ستون‌های متنی (object) =====')
object_cols = df.select_dtypes(include='object').columns.tolist()
print(object_cols)

Unnamed: 0                      int64
cat2_slug                      object
cat3_slug                      object
city_slug                      object
neighborhood_slug              object
                               ...   
rent_price_on_special_days    float64
rent_price_at_weekends        float64
location_latitude             float64
location_longitude            float64
location_radius               float64
Length: 61, dtype: object

===== ستون‌های متنی (object) =====
['cat2_slug', 'cat3_slug', 'city_slug', 'neighborhood_slug', 'created_at_month', 'user_type', 'description', 'title', 'rent_mode', 'rent_to_single', 'rent_type', 'price_mode', 'credit_mode', 'rent_credit_transform', 'transformable_price', 'deed_type', 'has_business_deed', 'floor', 'rooms_count', 'total_floors_count', 'unit_per_floor', 'has_balcony', 'has_elevator', 'has_warehouse', 'has_parking', 'construction_year', 'is_rebuilt', 'has_water', 'has_warm_water_provider', 'has_electricity', 'has_gas', 'has_heating_sy

In [16]:
numeric_suspects = ['floor', 'rooms_count', 'total_floors_count',
                    'unit_per_floor', 'construction_year', 'transformable_price']

for col in numeric_suspects:
    print(f'===== {col} =====')
    print(df[col].value_counts(dropna=False).head(10))
    print()

===== floor =====
floor
NaN    458252
2      128432
1      119495
3      109256
4       71885
5       37586
0       35834
6       13774
7        5708
-1       4462
Name: count, dtype: int64

===== rooms_count =====
rooms_count
دو              404050
یک              192083
NaN             154101
سه              138633
بدون اتاق        75898
چهار             21371
پنج یا بیشتر     13864
Name: count, dtype: int64

===== total_floors_count =====
total_floors_count
NaN    695648
4       89526
5       79911
3       52752
6       28877
2       22733
7        8271
8        4585
14       3046
10       2914
Name: count, dtype: int64

===== unit_per_floor =====
unit_per_floor
NaN            697717
2              119794
1               97712
4               36918
3               31423
6                4899
5                4811
8                3471
more_than_8      2083
7                 926
Name: count, dtype: int64

===== construction_year =====
construction_year
NaN     184172
۱۴۰۳    116260
۱

In [17]:
price_cols = ['price_value', 'price_mode', 'transformable_price',
              'credit_value', 'rent_value', 'transformable_credit',
              'rent_credit_transform']

for col in price_cols:
    print(f'===== {col} =====')
    print('نوع:', df[col].dtype)
    print(df[col].value_counts(dropna=False).head(5))
    print()

===== price_value =====
نوع: float64
price_value
NaN             431654
2.000000e+09      8974
2.500000e+09      8937
1.500000e+09      8934
3.000000e+09      8730
Name: count, dtype: int64

===== price_mode =====
نوع: object
price_mode
مقطوع     566444
NaN       426394
توافقی      5260
مجانی       1902
Name: count, dtype: int64

===== transformable_price =====
نوع: object
transformable_price
NaN      647106
False    279778
True      73116
Name: count, dtype: int64

===== credit_value =====
نوع: float64
credit_value
NaN            647905
100000000.0     34464
200000000.0     32508
300000000.0     25025
500000000.0     21040
Name: count, dtype: int64

===== rent_value =====
نوع: float64
rent_value
NaN          648678
0.0           59241
100000.0      24734
5000000.0     17475
3000000.0     16873
Name: count, dtype: int64

===== transformable_credit =====
نوع: float64
transformable_credit
NaN            647915
100000000.0     34463
200000000.0     32507
300000000.0     25024
500000000.0 

In [18]:
for col in ['floor', 'total_floors_count']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    print(f'{col}: نوع شد {df[col].dtype}')

floor: نوع شد float64
total_floors_count: نوع شد float64


In [19]:
for col in ['floor', 'total_floors_count']:
    print(f'===== {col} =====')
    print('تعداد NaN:', df[col].isnull().sum())
    print('min:', df[col].min(), '| max:', df[col].max())
    print()

===== floor =====
تعداد NaN: 458927
min: -1.0 | max: 30.0

===== total_floors_count =====
تعداد NaN: 696006
min: 2.0 | max: 30.0



In [20]:
rooms_map = {
    'بدون اتاق': 0,
    'یک': 1,
    'دو': 2,
    'سه': 3,
    'چهار': 4,
    'پنج یا بیشتر': 5  #پنج یا بیشتر به عدد ۵ نگاشت شد. این یک ساده‌سازی است (تمایز بالای ۵ اتاق از بین می‌رود)، اما چون این دسته از ابتدا در داده تجمیع‌شده بوده و تنها ۱.۴٪ داده‌هاست، تأثیر آن ناچیز است و رابطه‌ی ترتیبی حفظ می‌شود.
}

df['rooms_count'] = df['rooms_count'].map(rooms_map)

print('نوع:', df['rooms_count'].dtype)
print(df['rooms_count'].value_counts(dropna=False))

نوع: float64
rooms_count
2.0    404050
1.0    192083
NaN    154101
3.0    138633
0.0     75898
4.0     21371
5.0     13864
Name: count, dtype: int64


In [22]:
df['unit_per_floor'] = df['unit_per_floor'].replace('more_than_8', 9)
df['unit_per_floor'] = pd.to_numeric(df['unit_per_floor'], errors='coerce')

print('نوع:', df['unit_per_floor'].dtype)
print(df['unit_per_floor'].value_counts(dropna=False).head(12))

نوع: float64
unit_per_floor
NaN    697963
2.0    119794
1.0     97712
4.0     36918
3.0     31423
6.0      4899
5.0      4811
8.0      3471
9.0      2083
7.0       926
Name: count, dtype: int64


In [23]:
def fa_to_en_digits(text):
    if pd.isna(text):
        return text
    fa = '۰۱۲۳۴۵۶۷۸۹'
    en = '0123456789'
    table = str.maketrans(fa, en)
    return str(text).translate(table)

In [24]:
df['construction_year'] = df['construction_year'].apply(fa_to_en_digits)

temp = pd.to_numeric(df['construction_year'], errors='coerce')
non_numeric = df['construction_year'][temp.isna() & df['construction_year'].notna()]
print('مقادیر غیرعددی باقی‌مونده:')
print(non_numeric.value_counts())

مقادیر غیرعددی باقی‌مونده:
construction_year
قبل از 1370    20637
Name: count, dtype: int64


In [27]:
df['construction_year'] = df['construction_year'].replace('قبل از 1370', 1365)

df['construction_year'] = pd.to_numeric(df['construction_year'], errors='coerce')

print('نوع:', df['construction_year'].dtype)
print('تعداد NaN:', df['construction_year'].isnull().sum())
print('min:', df['construction_year'].min(), '| max:', df['construction_year'].max())
print(df['construction_year'].value_counts(dropna=False).head(10))

نوع: float64
تعداد NaN: 184172
min: 1365.0 | max: 1403.0
construction_year
NaN       184172
1403.0    116260
1390.0     59139
1402.0     58424
1400.0     53674
1395.0     53029
1398.0     38207
1397.0     36326
1396.0     35487
1401.0     35328
Name: count, dtype: int64


In [28]:
bool_cols = [
    'has_balcony', 'has_elevator', 'has_warehouse', 'has_parking',
    'has_security_guard', 'has_barbecue', 'has_pool', 'has_jacuzzi', 'has_sauna',
    'has_business_deed', 'is_rebuilt', 'has_warm_water_provider',
    'has_heating_system', 'has_cooling_system', 'has_restroom',
    'transformable_price', 'rent_credit_transform'
]

for col in bool_cols:
    print(f'{col}: {df[col].unique()}')

has_balcony: [nan 'true' 'false' 'unselect' True False]
has_elevator: [nan True False]
has_warehouse: [nan True False]
has_parking: [nan True False]
has_security_guard: [nan False True]
has_barbecue: [nan False True]
has_pool: [nan False True]
has_jacuzzi: [nan False True]
has_sauna: [nan False True]
has_business_deed: [nan False True]
is_rebuilt: [nan False True]
has_warm_water_provider: [nan 'package' 'water_heater' 'powerhouse' 'unselect']
has_heating_system: [nan 'shoofaj' 'duct_split' 'heater' 'split' 'fireplace' 'unselect'
 'floor_heating' 'fan_coil']
has_cooling_system: [nan 'air_conditioner' 'water_cooler' 'duct_split' 'split' 'unselect'
 'fan_coil']
has_restroom: [nan 'squat_seat' 'squat' 'seat' 'unselect']
transformable_price: [nan False True]
rent_credit_transform: [nan False True]


In [29]:
# بولین‌های تمیز
clean_bool_cols = [
    'has_elevator', 'has_warehouse', 'has_parking', 'has_security_guard',
    'has_barbecue', 'has_pool', 'has_jacuzzi', 'has_sauna', 'has_business_deed',
    'is_rebuilt', 'transformable_price', 'rent_credit_transform'
]

df['has_balcony'] = df['has_balcony'].replace({
    'true': True,
    'false': False,
    'unselect': np.nan
})

all_bool_cols = clean_bool_cols + ['has_balcony']

for col in all_bool_cols:
    df[col] = df[col].astype('boolean')

# چک
for col in all_bool_cols:
    print(f'{col}: {df[col].unique()}')

has_elevator: <BooleanArray>
[<NA>, True, False]
Length: 3, dtype: boolean
has_warehouse: <BooleanArray>
[<NA>, True, False]
Length: 3, dtype: boolean
has_parking: <BooleanArray>
[<NA>, True, False]
Length: 3, dtype: boolean
has_security_guard: <BooleanArray>
[<NA>, False, True]
Length: 3, dtype: boolean
has_barbecue: <BooleanArray>
[<NA>, False, True]
Length: 3, dtype: boolean
has_pool: <BooleanArray>
[<NA>, False, True]
Length: 3, dtype: boolean
has_jacuzzi: <BooleanArray>
[<NA>, False, True]
Length: 3, dtype: boolean
has_sauna: <BooleanArray>
[<NA>, False, True]
Length: 3, dtype: boolean
has_business_deed: <BooleanArray>
[<NA>, False, True]
Length: 3, dtype: boolean
is_rebuilt: <BooleanArray>
[<NA>, False, True]
Length: 3, dtype: boolean
transformable_price: <BooleanArray>
[<NA>, False, True]
Length: 3, dtype: boolean
rent_credit_transform: <BooleanArray>
[<NA>, False, True]
Length: 3, dtype: boolean
has_balcony: <BooleanArray>
[<NA>, True, False]
Length: 3, dtype: boolean


In [31]:
# unselect رو همه‌جا به NaN
df = df.replace('unselect', np.nan)

# ستون‌های دسته‌ای
category_cols = [
    'cat2_slug', 'cat3_slug', 'city_slug', 'neighborhood_slug', 'user_type',
    'rent_mode', 'rent_type', 'price_mode', 'credit_mode', 'deed_type',
    'building_direction', 'floor_material', 'property_type',
    'has_warm_water_provider', 'has_heating_system', 'has_cooling_system', 'has_restroom'
]

for col in category_cols:
    df[col] = df[col].astype('category')

# چک
print(df[category_cols].dtypes)

cat2_slug                  category
cat3_slug                  category
city_slug                  category
neighborhood_slug          category
user_type                  category
rent_mode                  category
rent_type                  category
price_mode                 category
credit_mode                category
deed_type                  category
building_direction         category
floor_material             category
property_type              category
has_warm_water_provider    category
has_heating_system         category
has_cooling_system         category
has_restroom               category
dtype: object


In [32]:
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])
    print('Unnamed: 0 حذف شد')
print('تعداد ستون‌ها:', df.shape[1])

Unnamed: 0 حذف شد
تعداد ستون‌ها: 60


In [33]:
print('نوع:', df['created_at_month'].dtype)
print(df['created_at_month'].value_counts(dropna=False).head(15))

نوع: object
created_at_month
2024-07-01 00:00:00    133219
2024-08-01 00:00:00    132396
2024-10-01 00:00:00    126387
2024-06-01 00:00:00    125624
2024-11-01 00:00:00    121456
2024-09-01 00:00:00    121432
2024-12-01 00:00:00    116452
2024-05-01 00:00:00    108759
2024-04-01 00:00:00      7187
2024-03-01 00:00:00      1881
2024-02-01 00:00:00      1211
2025-01-01 00:00:00      1139
2024-01-01 00:00:00       942
2023-12-01 00:00:00       532
2023-11-01 00:00:00       353
Name: count, dtype: int64


In [35]:
df['created_at_month'] = pd.to_datetime(df['created_at_month'], errors='coerce')

# تابع تبدیل میلادی به شمسی (به فرمت سال-ماه)
def to_shamsi_month(date):
    if pd.isna(date):
        return np.nan
    jd = jdatetime.date.fromgregorian(year=date.year, month=date.month, day=date.day)
    return f'{jd.year}-{jd.month:02d}'

df['created_at_shamsi'] = df['created_at_month'].apply(to_shamsi_month)

# چک
print(df['created_at_shamsi'].value_counts(dropna=False).head(15))

created_at_shamsi
1403-04    133219
1403-05    132396
1403-07    126387
1403-03    125624
1403-08    121456
1403-06    121432
1403-09    116452
1403-02    108759
1403-01      7187
1402-12      1881
1402-11      1211
1403-10      1139
1402-10       942
1402-09       532
1402-08       353
Name: count, dtype: int64


In [37]:
month_names = {
    1: 'فروردین', 2: 'اردیبهشت', 3: 'خرداد', 4: 'تیر',
    5: 'مرداد', 6: 'شهریور', 7: 'مهر', 8: 'آبان',
    9: 'آذر', 10: 'دی', 11: 'بهمن', 12: 'اسفند'
}

def to_shamsi_readable(date):
    if pd.isna(date):
        return np.nan
    jd = jdatetime.date.fromgregorian(year=date.year, month=date.month, day=date.day)
    return f'{month_names[jd.month]} {jd.year}'

df['created_at_shamsi_readable'] = df['created_at_month'].apply(to_shamsi_readable)

print(df['created_at_shamsi_readable'].value_counts(dropna=False).head(10))

created_at_shamsi_readable
تیر 1403         133219
مرداد 1403       132396
مهر 1403         126387
خرداد 1403       125624
آبان 1403        121456
شهریور 1403      121432
آذر 1403         116452
اردیبهشت 1403    108759
فروردین 1403       7187
اسفند 1402         1881
Name: count, dtype: int64


<div dir="rtl">

## مدیریت مقادیر گم‌شده

اول یه چیزو بگم: همه‌ی خالی‌ها یه جور نیستن. سه نوع خالی داریم و با هر کدوم فرق داره برخورد می‌کنیم:
- خالیِ ساختاری: فیلد برای این نوع آگهی اصلا معنی نداره (مثلا قیمت فروش واسه یه آگهی اجاره)
- خالی که یعنی «نداره»: کاربر تیک نزده (مثل امکانات)
- خالیِ واقعی: داده بوده ولی ثبت نشده

پس به جای اینکه کورکورانه همه‌ی خالی‌ها رو پر کنیم، واسه هر ستون جدا تصمیم گرفتیم. تصمیم‌ها هم بر اساس **معنیِ خالی‌بودن** بود، نه فقط درصدش.

**این ستونا رو حذف کردیم:**
- فیلدهای اجاره‌ی روزانه (`rent_to_single`, `cost_per_extra_person` و...): بالای ۹۷٪ خالین و فقط واسه آگهی‌های اجاره‌ی روزانه معنی دارن که خیلی کمن. هیچ جای پروژه هم لازم نیستن
- آب/برق/گاز (`has_water`, `has_electricity`, `has_gas`): فقط ۳.۴٪شون مقدار داشت و همون مقدارم نصف‌نصف True/False بود، یعنی به درد نمی‌خوره

**خالی این امکانات رو False گذاشتیم:** `has_parking`, `has_warehouse`, `has_balcony`, `has_elevator`, `has_security_guard`, `has_sauna`, `has_jacuzzi`, `has_pool`, `has_barbecue`
چون اگه کسی استخر یا آسانسور داشت، تقریبا همیشه ثبتش می‌کرد. پس نبودِ ثبت یعنی نداره.

**خالی اینا رو با میانه پر کردیم:** `construction_year`, `rooms_count`, `building_size`
چون عددی و معنادارن و خالیشون کمه. میانه رو به میانگین ترجیح دادیم چون به داده‌های پرت حساس نیست.

**چند ردیف خالی رو حذف کردیم:** `title`, `cat3_slug`, `city_slug` فقط چند تا (۵۴، ۱، ۲ تا) خالی داشتن، پس همون چند ردیفو حذف کردیم.

**اینا رو عمدا دست نزدیم (NaN موند):**
ستونای قیمت/اجاره/رهن، `has_business_deed`, `property_type`, `land_size`, `deed_type`, `rent_type`, `regular_person_capacity`, `transformed_rent`, `transformed_credit`.
اینا یا خالی‌بودنشون ساختاریه یا جای دیگه‌ی پروژه لازمن. پر کردنشون داده‌ی الکی می‌سازه، پس همون NaN موند تا هرکی لازم داشت خودش روی بخش پرش کار کنه. مخصوصا `regular_person_capacity` و `transformed_*` رو گذاشتیم واسه تصمیم بنیامین و تینا.

</div>

In [38]:
cols_to_drop = [
    # فیلدهای اجاره‌ی روزانه/اقامتگاهی
    'rent_to_single', 'cost_per_extra_person', 'rent_price_on_special_days',
    'rent_price_at_weekends', 'rent_price_on_regular_days', 'extra_person_capacity',
    # امکانات پایه (بی‌ارزش برای مدل‌سازی)
    'has_water', 'has_electricity', 'has_gas'
]

cols_to_drop = [c for c in cols_to_drop if c in df.columns]
df = df.drop(columns=cols_to_drop)

print('ستون‌های حذف‌شده:', cols_to_drop)
print('تعداد ستون باقی‌مونده:', df.shape[1])

ستون‌های حذف‌شده: ['rent_to_single', 'cost_per_extra_person', 'rent_price_on_special_days', 'rent_price_at_weekends', 'rent_price_on_regular_days', 'extra_person_capacity', 'has_water', 'has_electricity', 'has_gas']
تعداد ستون باقی‌مونده: 53


In [39]:
facility_cols = [
    'has_parking', 'has_warehouse', 'has_balcony', 'has_elevator',
    'has_security_guard', 'has_sauna', 'has_jacuzzi', 'has_pool', 'has_barbecue'
]

for col in facility_cols:
    df[col] = df[col].fillna(False)

print(df[facility_cols].isnull().sum())

has_parking           0
has_warehouse         0
has_balcony           0
has_elevator          0
has_security_guard    0
has_sauna             0
has_jacuzzi           0
has_pool              0
has_barbecue          0
dtype: int64


In [40]:
median_cols = ['construction_year', 'rooms_count', 'building_size']

for col in median_cols:
    median_value = df[col].median()
    df[col] = df[col].fillna(median_value)
    print(f'{col}: پُر شد با میانه = {median_value}')

construction_year: پُر شد با میانه = 1395.0
rooms_count: پُر شد با میانه = 2.0
building_size: پُر شد با میانه = 103.0


In [41]:
before = len(df)
df = df.dropna(subset=['title', 'cat3_slug', 'city_slug'])
after = len(df)

print(f'تعداد ردیف حذف‌شده: {before - after}')
print(f'تعداد ردیف باقی‌مونده: {after}')

تعداد ردیف حذف‌شده: 57
تعداد ردیف باقی‌مونده: 999943
